Setting

In [17]:
SEARCH_BEST_ESTIMATOR_XGBOOST = True
VERBOSE = True

In [18]:
import sys
sys.path.insert(0, "../My_RecSys_Course_AT_PoliMi")
import numpy as np
np.random.seed(1234)

import pandas as pd
import scipy.sparse as sps
path = "Data/data_train.csv"  # cartella da esplorare
# carica
df = pd.read_csv(path, low_memory=False)
userID_unique = df["row"].unique()
itemID_unique = df["col"].unique()
n_users = len(userID_unique)
n_items = len(itemID_unique)
data = np.ones(len(df))


URM_all = sps.csr_matrix((data, 
                          (df["row"].values, df["col"].values)),
                        shape = (n_users, n_items))


from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from Evaluation.Evaluator import EvaluatorHoldout
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_user_wise
from Recommenders.Recommender_import_list import *



#URM_train_val, URM_test = split_train_in_two_percentage_global_sample(URM_all, train_percentage = 0.8)
#URM_train, URM_validation = split_train_in_two_percentage_global_sample(URM_train_val, train_percentage = 0.8)
URM_train, URM_validation = split_train_in_two_percentage_global_sample(URM_all, train_percentage = 0.8)
evaluator_validation = EvaluatorHoldout(URM_validation, cutoff_list=[20])



EvaluatorHoldout: Ignoring 37 ( 0.1%) Users that have less than 1 test interactions


In [19]:
#from Recommenders.MatrixFactorization.Marco_MF_BPR_PyTorch import MF_BPR_PyTorch 

from Recommenders.MatrixFactorization.Marco_MF_BPR_PyTorch import MF_BPR_PyTorch
from Recommenders.MatrixFactorization.PyTorch.MF_MSE_PyTorch import MF_MSE_PyTorch
from Recommenders.Neural.MultVAE_PyTorch_Recommender import MultVAERecommender_PyTorch
from Recommenders.GraphBased.LightGCNRecommender import LightGCNRecommender
from Recommenders.GraphBased.challange2023_HHPRecommender import HHPRecommender
from Recommenders.GraphBased.challange2023_TwoWayRankAggregationRecommender import TwoWayRankAggregationRecommender


n_items = URM_train.shape[1]
p_dims = [252, n_items]

# Lista di configurazione con switch ON/OFF
algorithm_configs = [
    {
        "class": UserKNNCFRecommender,
        "label": "UserKNN",
        "load_model": True,
        "use": True,
        "fit_params": {'topK': 475, 'shrink': 5, 'similarity': 'cosine', 'normalize': True, 'feature_weighting': 'TF-IDF'} #0.22130051549598997
    },
    {
        "class": ItemKNNCFRecommender,
        "label": "ItemKNN",
        "load_model": True,
        "use": True,
        "fit_params": {'topK': 6, 'shrink': 125, 'similarity': 'jaccard', 'normalize': True, 'feature_weighting': 'TF-IDF'} #0.2192685676414274
    },
    {
        "class": RP3betaRecommender,
        "label": "RP3betaRecommender",
        "load_model": True,
        "use": True, 
        "fit_params": {'topK': 32, 'alpha': 1.5354226971328793, 'beta': 0.3756757967759691, 'normalize_similarity': True}  #0.2486049637145179
    },
    {
        "class": P3alphaRecommender,
        "label": "P3alpha",
        "load_model": True,
        "use": True,
        "fit_params": {'topK': 144, 'alpha': 1.7152863095489836, 'normalize_similarity': True} #0.2361672659432464
    },
    {
        "class": SLIM_BPR_Cython,
        "label": "SLIM_BPR",
        "load_model": True,
        "use": True, 
        "fit_params": {'epochs': 597, 'topK': 322, 'learning_rate': 0.07267995775964091, 'lambda_i': 7.651940663878026e-05, 'lambda_j': 0.0008278163477348147, 'symmetric': True} #0.2550437118081842
    },
    {
        "class": EASE_R_Recommender,
        "label": "EASE_R_Recommender",
        "load_model": True,
        "use": True, 
        "fit_params": {'l2_norm': 334.60181822439773, 'normalize_matrix': False, 'topK': 1000} #0.2822549018252873
    },
    {
        "class": MultiThreadSLIM_SLIMElasticNetRecommender,
        "label": "MultiThreadSLIM_SLIMElasticNetRecommender",
        "load_model": True,
        "use": True, 
        "fit_params": {'topK': 972, 'l1_ratio': 0.0001671165888065186, 'alpha': 0.003185833602582782} #0.28654878126942557
    },
    {
        "class": MF_BPR_PyTorch,
        "label": "MF_BPR_PyTorch",
        "load_model": True,
        "use": False, 
        "fit_params": {'num_factors': 141, 'learning_rate': 0.002732273917177109, 'batch_size': 256, 'lambda_reg': 2.1545713719004193e-05} #0.11098699211627468
    },
    {
        "class": MF_MSE_PyTorch,
        "label": "MF_MSE_PyTorch",
        "load_model": True,
        "use": True, 
        "fit_params": {"epochs":100, "batch_size":512}
    },  
    {
        "class": IALSRecommender,
        "label": "IALSRecommender",
        "load_model": True,
        "use": True, 
        "fit_params": {"num_factors": 50}
    },
    {
        "class": LightFMCFRecommender,
        "label": "LightFMCFRecommender",
        "load_model": True,
        "use": True, 
        "fit_params": {'n_components': 67, 'loss': 'warp', 'sgd_mode': 'adadelta', 'learning_rate': 3.1257391783478917e-06, 'item_alpha': 6.084721448115398e-05, 'user_alpha': 6.210347349694472e-05} #0.20986802421861195
    },
    {
        "class": MultVAERecommender_PyTorch,
        "label": "MultVAERecommender_PyTorch",
        "load_model": True,
        "use": True, 
        "fit_params": {'learning_rate': 0.0004999191960794948, 'l2_reg': 0.010941469935764224, 'dropout': 0.4556442393627423, 'total_anneal_steps': 103732, 'anneal_cap': 0.455538389778215, 'batch_size': 128, 'p_dims': p_dims, 'sgd_mode':'adam', 'epochs':300} #0.24612934627774993
    },
    {
        "class": HHPRecommender,
        "label": "HHP",
        "load_model": True,
        "use": True, 
        "fit_params": {'hybrid_lambda': 0.6104860363491649, 'topK': 30}
    },
    {
        "class": TwoWayRankAggregationRecommender,
        "label": "TWRA",
        "load_model": False,
        "use": False, 
        "fit_params": {
        "base_f_algorithm": "rp3beta",
        "base_b_algorithm": "rp3beta",
        "convex_lambda": 0.8912852477820357,
        
        # I parametri "f_..." vanno messi qui dentro, puliti dal prefisso
        "f_params": {
            "topK": 60,
            "alpha": 1.8420529491458713,
            "beta": 0.327990074964997,
            "normalize_similarity": True
        },
        
        # I parametri "b_..." vanno messi qui dentro, puliti dal prefisso
        "b_params": {
            "topK": 134,
            "alpha": 0.921144010150281,
            "beta": 0.6370917879824665,
            "normalize_similarity": False
        }
    }
    }
]

In [20]:
def hard_negative_sampling(
    user_id,
    negatives,
    models,
    hard_top_k=50,
    total_keep=200,
    temperature=1.0
):
    """
    Seleziona negativi privilegiando quelli con rank alto (hard negatives).
    VERSIONE VETTORIZZATA E VELOCE.
    """
    if len(negatives) <= total_keep:
        return negatives

    # 1. Convert to numpy for speed
    negatives_arr = np.array(negatives)
    
    # Init max scores with very low value
    max_scores = np.full(len(negatives), -np.inf)

    # 2. Vectorized Scoring
    for model in models:
        # Pass ALL negatives at once
        scores = model._compute_item_score([user_id], items_to_compute=negatives_arr)
        
        # Handle different output shapes
        if len(scores.shape) > 1:
            scores = scores[0]
        
        # Element-wise max to find the "best" score across models for each item
        max_scores = np.maximum(max_scores, scores)

    # 3. Sort by difficulty (High score = Hard negative)
    # We want indices that sort the array descending
    sorted_indices = np.argsort(max_scores)[::-1]

    # 4. HARD NEGATIVES PURI
    hard_indices = sorted_indices[:hard_top_k]

    # 5. SOFT SAMPLING PONDERATO
    remaining_indices = sorted_indices[hard_top_k:]
    
    final_indices = hard_indices

    if len(remaining_indices) > 0:
        n_soft = total_keep - len(hard_indices)
        n_soft = min(n_soft, len(remaining_indices))
        
        if n_soft > 0:
            # probabilità ∝ exp(-rank / T)
            # Lower rank index in 'remaining' means higher score -> higher prob
            ranks = np.arange(len(remaining_indices))
            probs = np.exp(-ranks / temperature)
            probs_sum = probs.sum()
            
            if probs_sum > 0:
                probs /= probs_sum
                
                sampled_soft_indices = np.random.choice(
                    remaining_indices,
                    size=n_soft,
                    replace=False,
                    p=probs
                )
                final_indices = np.concatenate([hard_indices, sampled_soft_indices])
            else:
                # Fallback uniform
                sampled_soft_indices = np.random.choice(
                    remaining_indices,
                    size=n_soft,
                    replace=False
                )
                final_indices = np.concatenate([hard_indices, sampled_soft_indices])

    # Convert back to item IDs
    return negatives_arr[final_indices].tolist()

Training/Loading

In [21]:
other_algorithms = {}

import os

output_folder_path = "result_models_URM_TRAIN/"

# Se la cartella non esiste, creala
if not os.path.exists(output_folder_path):
    os.makedirs(output_folder_path)

for config in algorithm_configs:
    print("-------------------------------------------------")
    # Controllo se l'utente vuole usare questo algoritmo
    if config["use"]:
        recommender_instance = config["class"](URM_train)
        file_path = os.path.join(output_folder_path, config["label"] + ".zip")

        if config["load_model"] and os.path.exists(file_path):
            print(f"Loading model {config['label']}...")
            recommender_instance.load_model(folder_path=output_folder_path, file_name=config["label"])
            print(f"Modello {config['label']} caricato da '{file_path}'")

        else:
            # fitta
            print(f"Training {config['label']}...")
            recommender_instance.fit(**config["fit_params"])
            recommender_instance.save_model(folder_path=output_folder_path, file_name=config["label"])
            print(f"Modello {config['label']} salvato in '{output_folder_path}'")
        
        # Aggiungi al dizionario 
        other_algorithms[config["label"]] = recommender_instance

    else:
        print(f"Skipping {config['label']} (use=False)")

print(f"\nModelli pronti per XGBoost: {list(other_algorithms.keys())}")

-------------------------------------------------
Loading model UserKNN...
UserKNNCFRecommender: Loading model from file 'result_models_URM_TRAIN/UserKNN'
UserKNNCFRecommender: Loading complete
Modello UserKNN caricato da 'result_models_URM_TRAIN/UserKNN.zip'
-------------------------------------------------
Loading model ItemKNN...
ItemKNNCFRecommender: Loading model from file 'result_models_URM_TRAIN/ItemKNN'
ItemKNNCFRecommender: Loading complete
Modello ItemKNN caricato da 'result_models_URM_TRAIN/ItemKNN.zip'
-------------------------------------------------
Loading model RP3betaRecommender...
RP3betaRecommender: Loading model from file 'result_models_URM_TRAIN/RP3betaRecommender'
RP3betaRecommender: Loading complete
Modello RP3betaRecommender caricato da 'result_models_URM_TRAIN/RP3betaRecommender.zip'
-------------------------------------------------
Loading model P3alpha...
P3alphaRecommender: Loading model from file 'result_models_URM_TRAIN/P3alpha'
P3alphaRecommender: Loading

Table creation

In [22]:
import pandas as pd
import numpy as np
import random, math
from tqdm import tqdm
import gc

candidate_models = [
    other_algorithms["RP3betaRecommender"],
    other_algorithms["IALSRecommender"],
    other_algorithms["EASE_R_Recommender"],
    other_algorithms["MultiThreadSLIM_SLIMElasticNetRecommender"],
    other_algorithms["MultVAERecommender_PyTorch"],
    #other_algorithms["TWRA"],
]
# --- CONFIGURAZIONE PER BAGGING ---
# 1. Cutoff Alto: Vogliamo trovare più item rilevanti possibili (Recall alta)
cutoff_per_model = 800 
#cutoff_default = 500
#cutoff_special = 100
#special_model_name = "MultiThreadSLIM_SLIMElasticNetRecommender"

# 2. Ratio Aumentato: Teniamo il 50-60% dei negativi ora (Master Pool).
# Poi ogni modello del bagging ne userà solo una parte (es. 20%).
# Se hai molta RAM (32GB+), puoi provare anche 0.8 o 1.0.
negative_sampling_ratio = 1.0

batch_size = 2000

print("Generazione candidati...")
print(f"Generazione candidati con cutoff {cutoff_per_model} e sampling {negative_sampling_ratio}...")

df_batches = []
total_batches = math.ceil(n_users / batch_size)

for start_idx in range(0, n_users, batch_size):
    end_idx = min(start_idx + batch_size, n_users)
    batch_users = range(start_idx, end_idx)
    
    users_list = []
    items_list = []
    labels_list = [] # Aggiungiamo subito la label!
    
    for user_id in tqdm(batch_users, leave=False, desc=f"Batch {start_idx//batch_size + 1}/{total_batches}"):
    # 1. RACCOLTA CANDIDATI
        current_user_candidates = set()
        for model in candidate_models:
            # LOGICA CUTOFF DIFFERENZIATO:
            # Verifichiamo se il modello è quello SLIM ElasticNet
            
            recs = model.recommend(user_id, cutoff=cutoff_per_model, remove_seen_flag=True)
            current_user_candidates.update(recs)
        
        candidates_list = list(current_user_candidates)
            
        # 2. IDENTIFICAZIONE POSITIVI (Ground Truth)
        start_pos = URM_validation.indptr[user_id]
        end_pos = URM_validation.indptr[user_id+1]
        # IMPORTANTE: Convertire in set per velocità (O(1) lookup)
        true_items = set(URM_validation.indices[start_pos:end_pos])
        
        # 3. SEPARAZIONE E CAMPIONAMENTO
        positives = []
        negatives = []
        
        for item in candidates_list:
            if item in true_items:
                positives.append(item)
            else:
                negatives.append(item)
        
        # Teniamo TUTTI i positivi
        final_items = positives.copy()
        final_labels = [True] * len(positives)
        
        # Campioniamo i negativi (ma ne teniamo di più rispetto a prima!)
        if len(negatives) > 0:
            n_keep = int(len(negatives) * negative_sampling_ratio)
            n_keep = max(n_keep, 50) # Minimo sindacale
            n_keep = min(n_keep, len(negatives))
            
            sampled_negatives = hard_negative_sampling(
                user_id=user_id,
                negatives=negatives,
                models=candidate_models,
                hard_top_k=50,        # sempre i 50 più difficili
                total_keep=n_keep,    # totale negativi da tenere
                temperature=0.7       # <1 = più focus sui difficili
            )

            final_items.extend(sampled_negatives)
            final_labels.extend([False] * len(sampled_negatives))

            
        # Aggiungiamo alle liste
        users_list.extend([user_id] * len(final_items))
        items_list.extend(final_items)
        labels_list.extend(final_labels)
    
    # Crea DataFrame Batch ottimizzato
    batch_df = pd.DataFrame({
        "UserID": np.array(users_list, dtype=np.int32),
        "ItemID": np.array(items_list, dtype=np.int32),
        "Label": np.array(labels_list, dtype=bool)
    })
    
    df_batches.append(batch_df)
    
    # Pulizia aggressiva
    del users_list, items_list, labels_list, batch_df, positives, negatives, final_items
    gc.collect()

print("Concatenazione finale...")
training_dataframe = pd.concat(df_batches, ignore_index=True)

print(f"DataFrame creato. Righe totali: {len(training_dataframe)}")
print(f"Positivi: {training_dataframe['Label'].sum()}")
print(f"Negativi: {(~training_dataframe['Label']).sum()}")


# Pulizia finale
del df_batches
gc.collect()

Generazione candidati...
Generazione candidati con cutoff 800 e sampling 1.0...


Concatenazione finale...
DataFrame creato. Righe totali: 51983482
Positivi: 568719
Negativi: 51414763


0

In [23]:
# Snippet da eseguire dopo aver creato 'training_dataframe'
# Conta quanti item "veri" (presenti in validation) sono finiti nel tuo dataframe di training
found_items = training_dataframe[training_dataframe["Label"] == True].shape[0]
total_items_in_validation = URM_validation.nnz

recall_candidates = found_items / total_items_in_validation
print(f"Recall massima teorica del set di candidati: {recall_candidates:.4f}")

Recall massima teorica del set di candidati: 0.9345


In [24]:
# Conta positivi (True) e negativi (False)
n_positives = training_dataframe["Label"].sum()
n_negatives = len(training_dataframe) - n_positives
ratio = n_negatives / n_positives if n_positives > 0 else 0

print(f"--- STATISTICHE DATASET ---")
print(f"Righe Totali: {len(training_dataframe)}")
print(f"Positivi (Items rilevanti): {n_positives}")
print(f"Negativi (Items irrilevanti): {n_negatives}")
print(f"Rapporto Negativi/Positivi: 1 : {ratio:.1f}")

# Se il rapporto è superiore a 1:100, l'XGBoost farà fatica.
# L'ideale per il training è portarlo tra 1:20 e 1:50.

print(f"Media candidati per utente: {len(training_dataframe) / n_users:.2f}")

--- STATISTICHE DATASET ---
Righe Totali: 51983482
Positivi (Items rilevanti): 568719
Negativi (Items irrilevanti): 51414763
Rapporto Negativi/Positivi: 1 : 90.4
Media candidati per utente: 1918.56


In [25]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
import gc

# Assicurati di aver già eseguito le celle che caricano URM_train, other_algorithms ed evaluator_validation

print("--- 1. Pre-Calcolo Dati Globali (Leggero) ---")

# 1. Popolarità Item e Utenti
# Usiamo csc/csr matrix per velocità
item_popularity = np.ediff1d(sps.csc_matrix(URM_train).indptr)
user_popularity = np.ediff1d(sps.csr_matrix(URM_train).indptr)

# 2. Clustering Utenti (su embedding IALS)
print("Fitting KMeans...")
# Recuperiamo i fattori latenti dall'IALS caricato
user_factors = other_algorithms['IALSRecommender'].USER_factors
n_clusters = 10  # Puoi provare anche 20
kmeans = KMeans(n_clusters=n_clusters, random_state=42).fit(user_factors)
user_clusters = kmeans.labels_          # Array: indice cluster per ogni utente
user_distances = kmeans.transform(user_factors) # Matrice: distanze dai centroidi

# 3. Popolarità per Cluster
print("Calcolo Popolarità Cluster...")
cluster_item_pop = np.zeros((n_clusters, n_items), dtype=np.float32)
coo_train = URM_train.tocoo()
# Iterazione veloce sugli elementi non-zero
for u, i in zip(coo_train.row, coo_train.col):
    c = user_clusters[u]
    cluster_item_pop[c, i] += 1.0

# Normalizziamo
for c in range(n_clusters):
    cluster_size = np.sum(user_clusters == c)
    if cluster_size > 0:
        cluster_item_pop[c, :] /= cluster_size

# 4. Performance per Cluster (MAP/Recall)
print("Calcolo Performance Cluster...")
cluster_perf_data = []

# Iteriamo sui cluster per vedere come performano i modelli base
for cluster_id in range(n_clusters):
    users_in_cluster = np.where(user_clusters == cluster_id)[0]
    # Intersezione con utenti di validazione
    valid_users_in_cluster = list(set(users_in_cluster) & set(evaluator_validation.users_to_evaluate))
    
    row_data = {'user_cluster': cluster_id}
    
    if len(valid_users_in_cluster) > 0:
        for name, model in other_algorithms.items():
            # Valuta il modello solo su questi utenti
            results_dict = evaluator_validation._run_evaluation_on_selected_users(model, users_to_evaluate=valid_users_in_cluster)
            # Salviamo la Recall@20 (o MAP)
            row_data[f'Cluster_RECALL_{name}'] = float(str(results_dict[20]["RECALL"]))
    else:
        # Fallback se il cluster non ha utenti nel validation set
        for name in other_algorithms.keys():
            row_data[f'Cluster_RECALL_{name}'] = 0.0
            
    cluster_perf_data.append(row_data)

cluster_perf_df = pd.DataFrame(cluster_perf_data)
# Convertiamo a float32 per risparmiare RAM
for col in cluster_perf_df.columns:
    if col != 'user_cluster':
        cluster_perf_df[col] = cluster_perf_df[col].astype('float32')

print("✅ Dati globali pronti!")

--- 1. Pre-Calcolo Dati Globali (Leggero) ---
Fitting KMeans...
Calcolo Popolarità Cluster...
Calcolo Performance Cluster...
✅ Dati globali pronti!


In [26]:
def add_features_to_batch(mini_df, models_dict):
    """
    Versione OTTIMIZZATA CON FIX DIMENSIONALE E SINTASSI CORRETTA
    """
    
    # Reset index per sicurezza
    mini_df = mini_df.reset_index(drop=True)
    num_rows = len(mini_df)
    
    # 1. PRE-ALLOCA ARRAY NUMPY
    model_scores = {name: np.zeros(num_rows, dtype=np.float32) for name in models_dict.keys()}
    
    # Array globali per accesso veloce
    item_ids_array = mini_df["ItemID"].values
    
    # 2. CALCOLO SCORE
    grouped = mini_df.groupby("UserID").indices
    
    for user_id, group_indices in grouped.items():
        # Preleva gli item ID corrispondenti a questi indici
        items_to_score = item_ids_array[group_indices]
        
        # Chiedi score ai modelli
        for name, model in models_dict.items():
            scores = model._compute_item_score([user_id], items_to_compute=items_to_score)
            
            # --- FIX DIMENSIONALE ---
            # Caso A: Il modello restituisce SOLO gli score richiesti
            if scores.shape[1] == len(items_to_score):
                model_scores[name][group_indices] = scores[0].flatten()
            
            # Caso B: Il modello restituisce TUTTI gli score (es. 6969 item)
            else:
                # Seleziona solo gli score degli item che ci interessano
                model_scores[name][group_indices] = scores[0, items_to_score]

    # 3. ASSEGNAZIONE AL DATAFRAME
    for name, values in model_scores.items():
        mini_df[name] = values

    # 4. FEATURE ENGINEERING
    model_columns = list(models_dict.keys())
    
    for col in model_columns:
        # Rank
        mini_df[f"{col}_rank"] = mini_df.groupby("UserID")[col].rank(ascending=False, method='min').astype('int32')
        
        # Norm (Min-Max per utente)
        user_max = mini_df.groupby("UserID")[col].transform("max")
        user_min = mini_df.groupby("UserID")[col].transform("min")
        denom = (user_max - user_min)
        denom[denom == 0] = 1 
        mini_df[f"{col}_norm"] = ((mini_df[col] - user_min) / denom).astype('float32')
        # Aggiungi questa trasformazione per ogni modello
        mini_df[f"{col}_rank_inv"] = 1.0 / (mini_df[f"{col}_rank"] + 1)
        safe_scores = np.maximum(mini_df[col], -0.999)
        mini_df[f"{col}_log"] = np.log1p(safe_scores).astype('float32')

    rank_cols = [f"{col}_rank" for col in model_columns]
    norm_cols = [f"{col}_norm" for col in model_columns]
    
    mini_df["vote_count_top5"] = (mini_df[rank_cols] <= 5).sum(axis=1).astype('int8')
    mini_df["vote_count_top10"] = (mini_df[rank_cols] <= 10).sum(axis=1).astype('int8')
    mini_df["vote_count_top15"] = (mini_df[rank_cols] <= 15).sum(axis=1).astype('int8')
    mini_df["vote_count_top20"] = (mini_df[rank_cols] <= 20).sum(axis=1).astype('int8')
    mini_df["vote_count_top30"] = (mini_df[rank_cols] <= 30).sum(axis=1).astype('int8')
    mini_df["vote_count_top100"] = (mini_df[rank_cols] <= 100).sum(axis=1).astype('int8')
    mini_df["mean_norm_score"] = mini_df[norm_cols].mean(axis=1).astype('float32')
    mini_df["std_norm_score"] = mini_df[norm_cols].std(axis=1).fillna(0).astype('float32')
    mini_df["max_norm_score"] = mini_df[norm_cols].max(axis=1).astype('float32')
    mini_df["min_norm_score"] = mini_df[norm_cols].min(axis=1).astype('float32')
    
    # 5. FEATURE GLOBALI
    idx_items = mini_df["ItemID"].values.astype(int)
    idx_users = mini_df["UserID"].values.astype(int)
    
    # Usa le variabili globali pre-calcolate
    mini_df['item_popularity'] = item_popularity[idx_items]
    mini_df['user_profile_len'] = user_popularity[idx_users]
    
    if "RP3betaRecommender_norm" in mini_df.columns and "SLIM_BPR_norm" in mini_df.columns:
        mini_df["diff_RP3beta_SLIM"] = (mini_df["RP3betaRecommender_norm"] - mini_df["SLIM_BPR_norm"]).astype('float32')
    else:
        mini_df["diff_RP3beta_SLIM"] = 0.0
    
    mini_df["std_rank"] = mini_df[rank_cols].std(axis=1).astype('float32')

    # 6. CLUSTERING
    if 'user_cluster' not in mini_df.columns:
        mini_df["user_cluster"] = user_clusters[idx_users]
    
    cluster_ids = mini_df["user_cluster"].values
    mini_df["cluster_pop"] = cluster_item_pop[cluster_ids, idx_items]
    
    dist_cols = [f"dist_cluster_{k}" for k in range(n_clusters)]
    mini_df[dist_cols] = user_distances[idx_users, :]
    
    # --- FIX SINTASSI QUI SOTTO (rimossi i backslash) ---
    mini_df = mini_df.merge(cluster_perf_df, on="user_cluster", how="left")
    
    epsilon = 1e-6
    sum_inverse = np.sum(1 / (mini_df[norm_cols] + epsilon), axis=1)
    mini_df["harmonic_mean_score"] = (len(norm_cols) / sum_inverse).astype('float32')
    
    return mini_df

XGBOOST

In [27]:
from xgboost import XGBRanker
from xgboost import XGBClassifier
from lightgbm import LGBMRanker
import xgboost as xgb

In [28]:
def sample_hard_negatives_for_bag(
    df_neg,
    n_needed,
    model_cols,
    hard_fraction=0.4,
    temperature=0.5,
    random_state=42
):
    """
    Campiona negativi privilegiando quelli difficili (rank alto).
    Usa SOLO feature già calcolate.
    """
    rng = np.random.RandomState(random_state)

    rank_cols = [f"{c}_rank" for c in model_cols]

    df_neg = df_neg.copy()

    # Hardness score (più basso = più difficile)
    df_neg["hardness"] = (
        df_neg[rank_cols].mean(axis=1)
        - 5.0 * df_neg["vote_count_top20"]
    )

    # Ordina per difficoltà
    df_neg = df_neg.sort_values("hardness")

    n_hard = int(n_needed * hard_fraction)
    n_soft = n_needed - n_hard

    # HARD NEGATIVES PURI
    hard_neg = df_neg.iloc[:n_hard]

    # SOFT NEGATIVES (sampling pesato)
    remaining = df_neg.iloc[n_hard:]

    if len(remaining) > 0 and n_soft > 0:
        ranks = np.arange(len(remaining))
        probs = np.exp(-ranks / temperature)
        probs /= probs.sum()

        soft_idx = rng.choice(
            remaining.index,
            size=min(n_soft, len(remaining)),
            replace=False,
            p=probs
        )
        soft_neg = remaining.loc[soft_idx]
        sampled = pd.concat([hard_neg, soft_neg])
    else:
        sampled = hard_neg

    return sampled.drop(columns=["hardness"])


In [ ]:
from xgboost import XGBRanker
from xgboost import XGBClassifier
from lightgbm import LGBMRanker
import xgboost as xgb
import numpy as np
import pandas as pd
import xgboost as xgb
from lightgbm import LGBMRanker
import gc



N_BAGS = 1          
NEGATIVES_PER_POSITIVE = 30 
HARD_NEG_FRACTION = 0.4     
bagged_models = []

model_cols = list(other_algorithms.keys())

features_to_use = (
    [f"{c}_rank" for c in model_cols] + 
    [f"{c}_norm" for c in model_cols] +
    ["vote_count_top5", "vote_count_top10", "vote_count_top15",
     "vote_count_top20", "vote_count_top30", "vote_count_top100",
     "mean_norm_score", "std_norm_score", "max_norm_score", "min_norm_score"] +
    ["item_popularity", "user_profile_len", "diff_RP3beta_SLIM", "std_rank"] +
    ["user_cluster", "cluster_pop"] + 
    [f"dist_cluster_{k}" for k in range(10)] 
)

df_pos_ids = training_dataframe[training_dataframe["Label"] == True]
df_neg_ids = training_dataframe[training_dataframe["Label"] == False]

print(f"Dataset Base: {len(df_pos_ids)} Positivi | {len(df_neg_ids)} Negativi totali.")

print("Pre-sorting negativi per Hardness (Item Popularity)...")

neg_item_ids = df_neg_ids["ItemID"].values.astype(int)
neg_item_pop = item_popularity[neg_item_ids]
neg_original_indices = df_neg_ids.index.values

sorted_args = np.argsort(neg_item_pop)[::-1]
sorted_neg_indices = neg_original_indices[sorted_args]

del neg_item_ids, neg_item_pop, neg_original_indices, sorted_args
gc.collect()

print(f"Inizio Training Iterativo di {N_BAGS} modelli...")

for bag_idx in range(N_BAGS):
    print(f"\n🚀 --- TRAINING BAG {bag_idx + 1}/{N_BAGS} ---")
    
    n_neg_needed = int(len(df_pos_ids) * NEGATIVES_PER_POSITIVE)
    n_neg_needed = min(n_neg_needed, len(df_neg_ids))
    
    n_hard = int(n_neg_needed * HARD_NEG_FRACTION)
    n_soft = n_neg_needed - n_hard

    # --- DIVERSITY FIX ---
    # Invece di prendere sempre i primi K (deterministic), prendiamo K casuali 
    # da una "pool" più grande dei top items (es. i primi 4*K)
    pool_size = min(len(sorted_neg_indices), n_hard * 4)
    hard_pool = sorted_neg_indices[:pool_size]
    
    # Random sampling dalla pool per garantire diversità tra bag
    np.random.seed(bag_idx * 1234 + 999) 
    hard_indices = np.random.choice(hard_pool, size=n_hard, replace=False)
    
    # I soft li prendiamo dal resto (oltre la pool)
    soft_candidates = sorted_neg_indices[pool_size:]
    
    if len(soft_candidates) > 0 and n_soft > 0:
        np.random.seed(bag_idx * 1234)
        soft_indices = np.random.choice(soft_candidates, size=min(n_soft, len(soft_candidates)), replace=False)
        final_indices = np.concatenate([hard_indices, soft_indices])
    else:
        final_indices = hard_indices

    current_neg_df = training_dataframe.loc[final_indices]
    
    mini_df = pd.concat([df_pos_ids, current_neg_df], axis=0, ignore_index=True)
    mini_df = mini_df.sort_values("UserID").reset_index(drop=True)
    
    n_pos_cur = mini_df["Label"].sum()
    n_neg_cur = len(mini_df) - n_pos_cur
    print(f"   📊 Statistiche Bag: {n_pos_cur} Positivi | {n_neg_cur} Negativi | Totale: {len(mini_df)}")

    del current_neg_df, final_indices, hard_indices
    if 'soft_indices' in locals(): del soft_indices
    gc.collect()

    print("   -> Calcolo Feature...")
    mini_df = add_features_to_batch(mini_df, other_algorithms)
    
    float_cols = mini_df.select_dtypes(include=['float64']).columns
    mini_df[float_cols] = mini_df[float_cols].astype('float32')

    X_bag = mini_df[features_to_use]
    y_bag = mini_df["Label"]
    groups_bag = mini_df.groupby("UserID").size().values.astype('int32')

    print("   -> Training XGBoost...")
    model = xgb.XGBRanker(
        learning_rate=0.066,
        max_depth=4,
        n_estimators=800,
        colsample_bytree=0.75,
        subsample=0.71,
        tree_method="hist", 
        random_state=42,
        objective="rank:pairwise",
        eval_metric="ndcg@20",
        n_jobs=-1
    )

    model.fit(X_bag, y_bag, group=groups_bag)
    bagged_models.append(model)
    print(f"   ✅ Modello XGBoost {bag_idx+1} completato!")

    print("   -> Training LGBM...")
    LGBM_model_final = LGBMRanker(
        learning_rate=0.05,
        num_leaves=95,
        n_estimators=105,
        random_state=42,
        verbose=-1,
        n_jobs=-1,
        objective="lambdarank",
        metric="ndcg",
        eval_at=20
    )

    LGBM_model_final.fit(X_bag, y_bag, group=groups_bag)
    bagged_models.append(LGBM_model_final)
    print(f"   ✅ Modello LGBM {bag_idx+1} completato!")

    del mini_df, X_bag, y_bag, groups_bag, model, LGBM_model_final
    gc.collect()

print("\n🎉 Tutti i modelli sono pronti!")

Dataset Base: 568719 Positivi | 51414763 Negativi totali.
Pre-sorting negativi per Hardness (Item Popularity)...
Inizio Training Iterativo di 3 modelli...

🚀 --- TRAINING BAG 1/3 ---
   📊 Statistiche Bag: 568719 Positivi | 17061570 Negativi | Totale: 17630289
   -> Calcolo Feature...
   -> Training XGBoost...
   ✅ Modello XGBoost 1 completato!
   -> Training LGBM...


/opt/anaconda3/envs/RSFramework/lib/python3.9/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


   ✅ Modello LGBM 1 completato!

🚀 --- TRAINING BAG 2/3 ---
   📊 Statistiche Bag: 568719 Positivi | 17061570 Negativi | Totale: 17630289
   -> Calcolo Feature...
   -> Training XGBoost...
   ✅ Modello XGBoost 2 completato!
   -> Training LGBM...


/opt/anaconda3/envs/RSFramework/lib/python3.9/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


   ✅ Modello LGBM 2 completato!

🚀 --- TRAINING BAG 3/3 ---
   📊 Statistiche Bag: 568719 Positivi | 17061570 Negativi | Totale: 17630289
   -> Calcolo Feature...
   -> Training XGBoost...
   ✅ Modello XGBoost 3 completato!
   -> Training LGBM...


/opt/anaconda3/envs/RSFramework/lib/python3.9/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


   ✅ Modello LGBM 3 completato!

🎉 Tutti i modelli sono pronti!


XGBOOST FINAL

# Retrain su tutto

In [30]:
other_algorithms_all = {}

import os

output_folder_path = "result_models_URM_ALL/"

# Se la cartella non esiste, creala
if not os.path.exists(output_folder_path):
    os.makedirs(output_folder_path)

for config in algorithm_configs:
    print("-------------------------------------------------")
    # Controllo se l'utente vuole usare questo algoritmo
    if config["use"]:
        recommender_instance = config["class"](URM_all)
        file_path = os.path.join(output_folder_path, config["label"] + ".zip")

        if config["load_model"] and os.path.exists(file_path):
            print(f"Loading model {config['label']}...")
            recommender_instance.load_model(folder_path=output_folder_path, file_name=config["label"])
            print(f"Modello {config['label']} caricato da '{file_path}'")

        else:
            # fitta
            print(f"Training {config['label']}...")
            recommender_instance.fit(**config["fit_params"])
            recommender_instance.save_model(folder_path=output_folder_path, file_name=config["label"])
            print(f"Modello {config['label']} salvato in '{output_folder_path}'")
        
        # Aggiungi al dizionario 
        other_algorithms_all[config["label"]] = recommender_instance

    else:
        print(f"Skipping {config['label']} (use=False)")

print(f"\nModelli pronti per XGBoost: {list(other_algorithms_all.keys())}")

-------------------------------------------------
Loading model UserKNN...
UserKNNCFRecommender: Loading model from file 'result_models_URM_ALL/UserKNN'
UserKNNCFRecommender: Loading complete
Modello UserKNN caricato da 'result_models_URM_ALL/UserKNN.zip'
-------------------------------------------------
Loading model ItemKNN...
ItemKNNCFRecommender: Loading model from file 'result_models_URM_ALL/ItemKNN'
ItemKNNCFRecommender: Loading complete
Modello ItemKNN caricato da 'result_models_URM_ALL/ItemKNN.zip'
-------------------------------------------------
Loading model RP3betaRecommender...
RP3betaRecommender: Loading model from file 'result_models_URM_ALL/RP3betaRecommender'
RP3betaRecommender: Loading complete
Modello RP3betaRecommender caricato da 'result_models_URM_ALL/RP3betaRecommender.zip'
-------------------------------------------------
Loading model P3alpha...
P3alphaRecommender: Loading model from file 'result_models_URM_ALL/P3alpha'
P3alphaRecommender: Loading complete
Mode

In [ ]:
# --- PREDIZIONE E SUBMISSION (ENSEMBLE DI BAGGING) ---
print("Generazione submission con ensemble di modelli...")
submission_data = []
target_users = pd.read_csv("Data/data_target_users_test.csv")["user_id"].values
batch_size = 500

for start_idx in tqdm(range(0, len(target_users), batch_size)):
    batch_users = target_users[start_idx : start_idx + batch_size]
    
    # 1. Genera candidati per il batch (ID)
    batch_users_list = []
    batch_items_list = []
    for u in batch_users:
        # Usa i candidati dai modelli base (cutoff alto)
        cands = set()
        for m in other_algorithms_all.values():
            cands.update(m.recommend(u, cutoff=1000, remove_seen_flag=True)) 
        for item in cands:
            batch_users_list.append(u)
            batch_items_list.append(item)
            
    batch_df = pd.DataFrame({"UserID": batch_users_list, "ItemID": batch_items_list})
    
    # 2. Calcola Feature
    batch_df = add_features_to_batch(batch_df, other_algorithms_all)
    
    # 3. Predizione Ensemble (Media dei modelli)
    X_test = batch_df[features_to_use]
    batch_df["score"] = 0.0
    
    for model in bagged_models:
        batch_df["score"] += model.predict(X_test)
    
    batch_df["score"] /= len(bagged_models)
    
    # 4. Selezione Top 20
    for u in batch_users:
        top_items = batch_df[batch_df["UserID"] == u].nlargest(20, "score")["ItemID"].values
        submission_data.append({"user_id": u, "item_list": " ".join(map(str, top_items))})

# Scrittura File
pd.DataFrame(submission_data).to_csv("submission_bagging.csv", index=False)
print("File creato: submission_bagging.csv")

Generazione submission con ensemble di modelli...


  0%|          | 0/55 [00:00<?, ?it/s]/opt/anaconda3/envs/RSFramework/lib/python3.9/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/opt/anaconda3/envs/RSFramework/lib/python3.9/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/opt/anaconda3/envs/RSFramework/lib/python3.9/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
  2%|▏         | 1/55 [00:27<24:30, 27.23s/it]/opt/anaconda3/envs/RSFramework/lib/python3.9/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'ev

File creato: submission_bagging.csv
